# Retrieval tests — ISLP

Goal:

1. **(a)** verify hybrid retrieval (dense + BM25 fused via RRF) returns relevant sections;
2. **(b)** verify metadata-filtered queries return correctly scoped results;
3. **(c)** verify the parent-document pattern returns **full sections** rather than tiny child chunks.

**Prerequisite:** chapter has already been ingested — see `02_ingest_islp_ch2.ipynb`.

> **Note:** Backed by Qdrant. Dashboard: http://localhost:6333/dashboard.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from src.services.retrieval.retrievers import build_retriever
retriever = build_retriever(book_slug="islp")

## 1. Data query — semantic content match

Query the bias-variance tradeoff. We expect to surface **section 2.2.2** of ISLP.

In [ ]:
query = "explain the bias-variance tradeoff in supervised learning"
results = retriever.invoke(query)
for i, d in enumerate(results, 1):
    print(f"[{i}] {d.metadata['h2_path']}  (book={d.metadata['book_slug']}, pages={d.metadata.get('page_from')}-{d.metadata.get('page_to')})")
    print("    parent length:", len(d.page_content), "chars")
    print("    has_formula:", d.metadata.get("has_formula"))
    print()

## 2. Verify parent (full section) returned, not child

Parent docs are whole sections (up to `parent_max_chars`), not 800-char children. Typical length should be well above 800 chars.

In [ ]:
print("Average parent length:", sum(len(d.page_content) for d in results) / len(results))
print("\nFirst 600 chars of top result:")
print(results[0].page_content[:600], "...")

## 3. Metadata filter — only sections containing formulas

Bypass the retriever and hit Qdrant directly so we can apply scalar payload filters.

In [ ]:
from src.core.qdrant_store import client
from src.core.config import settings
from qdrant_client.models import Filter, FieldCondition, MatchValue

qc = client()

flt = Filter(must=[
    FieldCondition(key="book_slug", match=MatchValue(value="islp")),
    FieldCondition(key="has_formula", match=MatchValue(value=True)),
])
res, _ = qc.scroll(settings.qdrant_collection_text, scroll_filter=flt, limit=10, with_payload=True)
for p in res[:5]:
    print(p.payload.get("h2_path"), "n_formulas=", p.payload.get("n_formulas"))

## 4. Metadata filter — only subsections under section 2.2

Scope to chapter 2 and inspect which `h2_path` values come back.

In [ ]:
flt = Filter(must=[
    FieldCondition(key="book_slug", match=MatchValue(value="islp")),
    FieldCondition(key="chapter_id", match=MatchValue(value="ch02")),
])
res, _ = qc.scroll(settings.qdrant_collection_text, scroll_filter=flt, limit=10, with_payload=True)
seen_sections = sorted({p.payload.get("h2_path") for p in res if p.payload.get("h2_path")})
for h in seen_sections:
    print(" -", h)

## 5. End-to-end chain — answer + sources

Uses `src.retrieval.chain.build_chain`, which wires retriever → prompt → LLM and returns both an answer and the source documents.

In [ ]:
from src.services.retrieval.chain import build_chain
chain = build_chain()
result = chain.invoke("What is the difference between supervised and unsupervised learning?")
print("ANSWER:")
print(result["answer"])
print("\nSOURCES:")
for s in result["sources"]:
    print(" -", s.metadata.get("h2_path"), "pages", s.metadata.get("page_from"))

## 6. Sanity check — synopsis surfaced via metadata

Confirm that the LLM-generated synopsis travels with each chunk as a scalar metadata field.

In [ ]:
results = retriever.invoke("K-nearest neighbors classification")
for r in results[:3]:
    print("Section:", r.metadata["h2_path"])
    print("Synopsis:", r.metadata.get("synopsis", "")[:200])
    print()

## Notes

- **Hybrid retrieval:** dense embeddings + BM25 (sparse) are fused with Reciprocal Rank Fusion via Qdrant's native `FusionQuery(Fusion.RRF)` over named vectors (`text` dense + `bm25` sparse).
- **Parent-document pattern:** child chunks (~800 chars) drive retrieval for precision, but each hit resolves back to its parent section via `parent_id`, so the LLM sees full context.
- **Metadata filters:** Qdrant supports scalar payload filters (`book_slug`, `chapter_id`, `has_formula`, …) via `Filter` / `FieldCondition`, flattened by `build_documents._flatten_meta`.
- **Synopsis:** stored as a (truncated) scalar string in the Qdrant point payload so it can be inspected without rehydrating parents.

In [ ]:
from src.services.retrieval.retrievers import search_images
hits = search_images("wage vs age figure", book_slug="islp", k=3)
for h in hits:
    print(h["image_name"], "-", h["image_reference"][:120])